In [1]:
# 读取数据
library(data.table)
library(grf)

dir <- "/root/autodl-tmp/DATA/R-DATA"
out_dir <- "/root/autodl-tmp/Table7_and_Figure6-8"

# 若目录不存在则创建
if (!dir.exists(out_dir)) dir.create(out_dir, recursive = TRUE)

set1_train <- fread(file.path(dir, "set1_train.csv.gz"))
set1_test  <- fread(file.path(dir, "set1_test.csv.gz"))

# set2_train <- fread(file.path(dir, "set2_train.csv.gz"))
# set2_test  <- fread(file.path(dir, "set2_test.csv.gz"))

# set3_train <- fread(file.path(dir, "set3_train.csv.gz"))
# set3_test  <- fread(file.path(dir, "set3_test.csv.gz"))

set.seed(20240229)

# 抽样规模
N_TRAIN_DEV <- 100000
N_TEST_DEV  <- 30000

# 分层抽样函数（保持不变）
sample_stratified <- function(dt, ycol, n_total) {
  dt0 <- dt[get(ycol) == 0]
  dt1 <- dt[get(ycol) == 1]
  
  p1 <- nrow(dt1) / nrow(dt)
  n1 <- max(2000, as.integer(n_total * p1))
  n0 <- n_total - n1
  
  rbind(
    dt0[sample.int(nrow(dt0), min(n0, nrow(dt0)))],
    dt1[sample.int(nrow(dt1), min(n1, nrow(dt1)))]
  )
}

# 数据列表
# train_list <- list(set1 = set1_train, set2 = set2_train, set3 = set3_train)
# test_list  <- list(set1 = set1_test,  set2 = set2_test,  set3 = set3_test)

train_list <- list(set1 = set1_train)
test_list  <- list(set1 = set1_test)

# 参数
Y_COL <- "insider_trading"
ID_COLS <- c("Stkcd", "Trddt")
# W_COLS <- c("AQsp_Amount_lag1", "AQsp_Volume_lag1", "AQsp_time_lag1")
W_COLS <- c("AQsp_Amount_lag1")
# 结果容器
result_list <- list()

for (set_name in names(train_list)) {
  
  cat("\n==========", set_name, "==========\n")
  
  train_dev <- sample_stratified(train_list[[set_name]], Y_COL, N_TRAIN_DEV)
  test_dev  <- sample_stratified(test_list[[set_name]],  Y_COL, N_TEST_DEV)
  
  train_dev <- train_dev[sample.int(.N)]
  test_dev  <- test_dev[sample.int(.N)]
  
  dt <- train_dev
  
  TREAT_COLS <- grep("_lag1$", names(dt), value = TRUE)
  
  for (W_COL in W_COLS) {
    
    cat("Running:", set_name, "|", W_COL, "\n")
    
    X_COLS <- setdiff(names(dt), c(Y_COL, ID_COLS, TREAT_COLS))
    
    Y <- dt[[Y_COL]]
    W <- dt[[W_COL]]
    X <- as.matrix(dt[, ..X_COLS])
    
    t0 <- Sys.time()
    
    cf_dev <- causal_forest(
      X, Y, W,
      num.trees = 5000,
      sample.fraction = 0.3,
      min.node.size = 50,
      honesty = TRUE,
      seed = 20240229
    )
    
    t1 <- Sys.time()
    dt_sec <- as.numeric(difftime(t1, t0, units = "secs"))
    
    # =========================
    # 保存模型（关键修改点）
    # =========================
    model_file <- file.path(
      out_dir,
      paste0("cf_model_", set_name, "_", W_COL, ".rds")
    )
    saveRDS(cf_dev, model_file)
    
    # =========================
    # 计算ATE
    # =========================
    ate_res <- average_treatment_effect(cf_dev)
    
    ate_hat <- ate_res[["estimate"]]
    se_hat  <- ate_res[["std.err"]]
    
    ci_low  <- ate_hat - 1.96 * se_hat
    ci_high <- ate_hat + 1.96 * se_hat
    
    # =========================
    # 存结果
    # =========================
    result_list[[paste0(set_name, "_", W_COL)]] <- data.table(
      dataset = set_name,
      treatment = W_COL,
      ate = ate_hat,
      se = se_hat,
      ci_low = ci_low,
      ci_high = ci_high,
      runtime_sec = dt_sec,
      model_path = model_file
    )
    
    print(result_list[[paste0(set_name, "_", W_COL)]])
  }
}

# 汇总结果
result_dt <- rbindlist(result_list)
# 导出用结果
result_export <- result_dt[, .(
  dataset,
  treatment,
  ate,
  se,
  runtime_sec
)]

result_file <- file.path(out_dir, "Table7_ATE_results.csv")
fwrite(result_export, result_file)
print(result_export)


========== set1 ==========
Running: set1 | AQsp_Amount_lag1 
   dataset        treatment         ate          se       ci_low     ci_high
    <char>           <char>       <num>       <num>        <num>       <num>
1:    set1 AQsp_Amount_lag1 0.003334109 0.002768564 -0.002092277 0.008760495
   runtime_sec
         <num>
1:    1122.646
                                                                 model_path
                                                                     <char>
1: /root/autodl-tmp/Table7_and_Figure6-8/cf_model_set1_AQsp_Amount_lag1.rds
   dataset        treatment         ate          se runtime_sec
    <char>           <char>       <num>       <num>       <num>
1:    set1 AQsp_Amount_lag1 0.003334109 0.002768564    1122.646
